# 0

## 准备工作

In [17]:
import os
from dotenv import load_dotenv
from langchain_community.graphs import Neo4jGraph
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document
from pyvis.network import Network
from langchain.chat_models import init_chat_model
import pandas as pd
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import PromptTemplate
from typing import Tuple, Optional
import re
import numpy as np
from sentence_transformers import SentenceTransformer, util

from dotenv import load_dotenv
import asyncio

os.environ["DEEPSEEK_API_KEY"] = "sk-ae6d423f13b7403fa8ba16ddb4acc297"
os.environ["QWEN_API_KEY"] = "sk-9b4b258c64544e609ff95dae918a401f" 

# 浏览器端口：7476
url = "os.getenv('NEO4J_URL')"
username = "neo4j"
password = "12345678"

In [2]:
# 1.读取neo4j数据库
load_dotenv()
graph = Neo4jGraph(
    url=url,
    username=username,
    password=password,
)

C:\Users\lenovo\AppData\Local\Temp\ipykernel_32640\3089890364.py:3: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(


In [3]:
# 2.定义LLM
from langchain.chat_models import init_chat_model

llm_ds=init_chat_model(
    model='deepseek:deepseek-chat',
    temperature=1.3)

llm_qwen=ChatOpenAI(
    model="qwen-max",
    openai_api_key=os.getenv("QWEN_API_KEY"),
    openai_api_base="https://dashscope.aliyuncs.com/compatible-mode/v1",
    temperature=1.3
)

## 1 基础版生成 

In [4]:
# 3.图检索
#基于cypher
#返回子图
def retrieve_1hop_subgraph(graph, keyword):
    query = """
    MATCH (n {id: $kw})
    OPTIONAL MATCH (n)-[r]-(m)
    RETURN 
        COLLECT(DISTINCT n) + COLLECT(DISTINCT m) AS nodes,
        COLLECT(DISTINCT r) AS relationships
    """
    res = graph.query(query, {"kw": keyword})
    if not res or not res[0]["nodes"]:
        return {"nodes": [], "relationships": []}
    return {
        "nodes": res[0]["nodes"],
        "relationships": res[0]["relationships"]
    }

In [5]:
# 4.从图构建上下文
# 检索出的子图数据（即相关知识）-->文本，作为生成的知识上下文
def get_knowledge_context(subgraph):
    sentences = set()
    
    for rel in subgraph["relationships"]:
        source = rel[0]['id']
        target = rel[2]['id']
        rtype = rel[1]
        # 转换为自然语言句子
        sentence = f"{source}{rtype}{target}。"
        sentences.add(sentence)
    
    return "\n".join(sorted(sentences))

In [81]:
# 5.提示词
from langchain_core.prompts import PromptTemplate
answer_prompt=PromptTemplate.from_template(
    '''
    你是一名高中地理资深命题教师。
    
    【已知条件】
    - 知识上下文：
    {context}
    - 目标知识点： {knowledge_point}
    - 难度等级：{difficulty}
    - 题型：{q_type}
    - 是否涉及运算：{calculation}
    - 认知层级（如：记忆 / 理解 / 应用 / 分析 / 评价 / 创造）：
    {cognitive_level}
    
    【出题要求】
    1. 题目必须以「目标知识点」为核心  
    2. 可合理使用上下文中的相关知识点作为支撑，但只能起辅助作用  
    3. 严格匹配给定的题型、难度、是否运算、认知层级
    4. 题干表述清晰，不引入无关信息。纯文本题，不要用到图片。
    5. 设问方式灵活，所问的可能是下一句或者选择一个陈述句（这两种情况最多），也可能是挖空，也可能是从下列选择或排序等
    5. 有概率用真实地理景观为例出题，但不要只用上下文中的，可以用其他的，比如“乞拉朋齐……据此请分析……的成因主要是”
    5. 只生成一道题，不要解释、不输出思路
    6. 要符合高中地理考题风格

    严格按照以下格式（包括换行也要一致）输出，不要含有其他内容如前后缀等，最前面不要有空格
    【输出格式】：
    【{difficulty}题 | {cognitive_level}题 | 运算题（如果涉及运算的话）】前两个xx分别填难度、认知层级
    【题目】
    （另起一行）题目内容
    （如是选择题）【选项】
    【参考答案】
    （另起一行）（如果是选择题，只输出选项一个字母）
    '''
)

In [83]:
# 检索-生成pipeline
def rag(knowledge_point,difficulty,
        q_type,calculation,cognitive_level,prompt):
    # 检索出的子图
    graph_results=retrieve_1hop_subgraph(graph,knowledge_point)
    # 子图->知识上下文
    context=get_knowledge_context(graph_results)
    response=llm.invoke(
        prompt.format(
            context=context,
            knowledge_point=knowledge_point,
            difficulty=difficulty,
            q_type=q_type,
            calculation=calculation,
            cognitive_level=cognitive_level
        )
    )

    return graph_results,context,response.content

In [86]:
# 检索-生成pipeline
# react修改专用
def rag_revise(knowledge_point,previous_question,
        action,feedback,prompt):
    # 检索出的子图
    graph_results=retrieve_1hop_subgraph(graph,knowledge_point)
    # 子图->知识上下文
    context=get_knowledge_context(graph_results)
    response=llm.invoke(
        prompt.format(
            previous_question=previous_question,
            action=action,
            feedback=feedback,
            knowledge_point=knowledge_point
        )
    )

    return graph_results,context,response.content

In [8]:
# 试题库特征建模，存成字典
# 以下的block表示小节
block2id={'流水地貌':0,'风成地貌':1,'喀斯特、海岸和冰川地貌':2}
feature_map={
    '流水地貌':{
        'k_points':["地貌", "流水地貌", "流水侵蚀地貌", "流水堆积地貌", "峡谷", "河漫滩", "河流阶地", "曲流", "牛轭湖", "冲积扇", "洪积扇", "冲积平原", "三角洲", "江心洲", "滑坡", "泥石流", "流水", "山洪", "虎跳峡", "雅鲁藏布大峡谷", "嘉陵江", "崇明岛", "橘子洲", "黄河三角洲", "尼罗河三角洲", "华北平原", "东北平原", "长江中下游平原"],
        'difficulty':[33/43,8/43,2/43],
        'q_type':[1], 
        'calculation':0.0,
        'cognitive_levels':[5/43,30/43,0/43,8/43,0/43,0/43]
    },
    '风成地貌':{
        'k_points':["风蚀地貌", "风积地貌", "风蚀蘑菇", "风蚀壁龛", "风蚀柱", "雅丹地貌", "沙丘", "新月形沙丘", "灌丛沙丘", "风力"],
        'difficulty':[13/23,10/23,0/23],
        'q_type':[1], 
        'calculation':0.0,
        'cognitive_levels':[12/23,7/23,1/23,3/23,0/23,0/23]
    },
    '喀斯特、海岸和冰川地貌':{
        'k_points':["喀斯特地貌", "海岸地貌", "冰川地貌", "海蚀地貌", "海积地貌", "海蚀崖", "海蚀平台", "海蚀柱", "海滩", "岬角", "冰斗", "冰川槽谷", "角峰", "刃脊", "峡湾", "波浪", "冰川", "云南石林", "广西桂林", "重庆奉节小寨天坑", "贵州荔波", "四川黄龙", "湖南张家界黄龙洞", "澳大利亚坎贝尔港国家公园", "挪威西峡湾"],
        'difficulty':[17/37,1/37,19/37],
        'q_type':[1], 
        'calculation':0.0,
        'cognitive_levels':[15/37,4/37,2/37,16/37,0/37,0/37]
    }
}

In [9]:
# 学生背景，存成字典
stu_map=[
    {'handle':0.5,
     'calculation':0.7,
     'cognitive_levels':3
    },
    {'handle':0.2,
     'calculation':0.5,
     'cognitive_levels':2
    },
    {'handle':0.7,
     'calculation':0.5,
     'cognitive_levels':4
    }
     ]

In [10]:
# 试题特征融合
from math import exp

def merge_feature(block,stu_group):
    # 0.取出当前学生的背景、当前小节的题库分布
    # 假设细胞外液知识点 属于 生理指标小节
    # block_id=block2id[block]
    true_feature=feature_map[block]
    stu_bg=stu_map[stu_group]

    # 1.难度
    t=0.5 #温度系数
    difficulty_spot=[0.2,0.5,0.85] #三个难度对应的标准难度系数
    difficulty=true_feature['difficulty']
    difficulty_p=[] #最后的难度概率分布
    sum_p=0
    for i in range(3):
        distance=abs(difficulty[i]-difficulty_spot[i])
        w_i=exp(-distance/t)
        p=difficulty[i]*w_i
        difficulty_p.append(p)
        sum_p+=p
    for i in range(3):  #归一化
        difficulty_p[i]=difficulty_p[i]/sum_p
        
    # 2.涉及运算概率
    # 1.5的来历是设运算能力=0.5为中等值，这样当运算能力>0.5，运算题概率会被缩小
    calculation=true_feature['calculation']*(1.5-stu_bg['calculation'])
    
    # 3.认知层级
    decay1=[0.3,0.4,0.5,0.7,0.8,1]
    decay2=[1,0.5,0.25,0.1,0.1,0.1]
    cognitive_levels=true_feature['cognitive_levels']
    c=stu_bg['cognitive_levels']
    cognitive_p=[0 for _ in range(6)]
    for i in range(c+1):
        cognitive_p[i]=cognitive_levels[i]*decay1[6-c+i-1]
    if c<5:
        for i in range(c+1,6):
            cognitive_p[i]=cognitive_levels[i]*decay2[i-c]
    sum_p=sum(cognitive_p)
    for i in range(6):
        cognitive_p[i]=cognitive_p[i]/sum_p

    # print(true_feature)
    # print(stu_bg)
    # print(difficulty_p)
    # print(calculation)
    # print(cognitive_p)

    return difficulty_p,calculation,cognitive_p

In [11]:
# 采样 获取单次生成的具体取值
import random
difficulty_list=['简单','中等','困难']
q_type_list=['选择题']
calculation_list=['不涉及','涉及']
cognitive_levels=['记忆','理解','应用','分析','评价','创造']

def choose_feature(difficulty_p,calculation,cognitive_p):
    # 1. 抽难度（0:简单 1:中等 2:困难）
    cur_difficulty = random.choices([0, 1, 2], weights=difficulty_p, k=1)[0]
    cur_difficulty=difficulty_list[cur_difficulty]
    # 2. 抽是否运算（0/1）
    cur_calculation = random.random() < calculation
    cur_calculation=calculation_list[cur_calculation]
    # 3. 抽认知层级（0~5）
    cur_cognitive_level = random.choices(range(6), weights=cognitive_p, k=1)[0]
    cur_cognitive_level=cognitive_levels[cur_cognitive_level]
    
    print(cur_difficulty,cur_calculation,cur_cognitive_level)
    return cur_difficulty,cur_calculation,cur_cognitive_level

In [84]:
# 基于特征采样的生成
# knowledge_point可以不指定（只指定小节block），也可以指定
def get_questions(block,stu_group,knowledge_point=None,revise=False):
    # 抽取生成要求
    difficulty_p,calculation,cognitive_p=merge_feature(
        block=block,
        stu_group=stu_group)
    
    cur_difficulty,cur_calculation,cur_cognitive_level=choose_feature(
        difficulty_p,calculation,cognitive_p
    )
    # 抽取知识点
    all_kps=feature_map[block]['k_points']
    if not knowledge_point or knowledge_point not in allkps:
        knowledge_point=random.choices(all_kps,k=1)[0]
    # 生成
    graph_results,context,answer=rag(
            knowledge_point=knowledge_point,
            difficulty=cur_difficulty,
            q_type=q_type_list[0],
            calculation=cur_calculation,
            cognitive_level=cur_cognitive_level,
            prompt=prompt
        )
        
    features=[knowledge_point,cur_difficulty,q_type_list[0],
              cur_calculation,cur_cognitive_level,]

    return answer,features,context

In [109]:
# 解析llm的返回->结构化
def parse(answer:str,features,context):
    # print(answer)
    lines = [l.strip() for l in answer.strip().splitlines() if l.strip()]
    # print(lines)
    # 题干、选项、参考答案
    q_start = lines.index("【题目】") + 1
    o_start = lines.index("【选项】")
    a_start = lines.index("【参考答案】") + 1

    question = " ".join(lines[q_start:o_start])
    options = " ".join(lines[o_start + 1:a_start - 1])
    answer_text = lines[a_start]

    context = context.replace('\n', '')

    features.append(question)
    features.append(options)
    features.append(answer_text)
    features.append(context)
    return features

In [14]:
# 单次生成（采样-检索-生成-解析）
# 预期的含决策+记忆的生成pipeline: 循环 [采样-检索构造上下文-提示词记忆注入（长期->短期）-生成]->评估->短期记忆更新 -> 决策 —> 循环 or PASS  // ——> 长期记忆更新


In [ ]:
# 批量生成
llm = llm_ds
n = 500
save_path = './output/ds-210-instruction-data.txt'

count = 0  # 成功生成的题目数
attempt = 0  # 总尝试次数（防死循环）

with open(save_path, "w", encoding="utf-8") as f:
    while count < n and attempt < n * 3:  # 最多尝试 3n 次
        attempt += 1
        print(f'正在生成第{count+1}题（尝试第{attempt}次）：', end=' ')
        try:
            answer, features, context = get_questions(
                block='流水地貌',
                stu_group=0,
                knowledge_point=None
            )
            features = parse(answer, features, context)  # 这里用 count 做题号
            f.write(";".join(features) + "\n")
            f.flush()  # 立即写入，避免丢失
            count += 1
            print('成功')
        except Exception as e:
            print(f'【生成有误，重试】错误信息：{e}')

    print(f'生成完成，共成功 {count} 题，保存至 {save_path}')

## 2 评估器

## 2.1 知识点相关性

In [18]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')

In [19]:
def build_full_question(row):
    """
    根据字段拼合完整试题（题干 + 正确答案文本）
    输入 row: dict，包含字段：
        '题干', '选项', '正确答案'
    输出: 完整试题字符串
    """
    question = row['题干'].strip()
    answer_raw = row['正确答案'].strip()
    options_text = row['选项'].strip()

    # 1. 尝试从选项字段中提取正确答案对应的文本（选择题）
    answer_text = answer_raw
    if re.match(r'^[A-D]$', answer_raw) and options_text:
        # 按 "A." "B." 等分割，简单提取
        parts = re.split(r'[A-D]\.', options_text)
        # parts[0] 是空，parts[1] 对应 A 选项，parts[2] 对应 B，依次类推
        opt_dict = {}
        opt_letters = re.findall(r'([A-D])\.', options_text)
        opt_contents = re.split(r'[A-D]\.', options_text)[1:]  # 去掉首空
        for letter, content in zip(opt_letters, opt_contents):
            opt_dict[letter] = content.strip()
        if answer_raw in opt_dict:
            answer_text = opt_dict[answer_raw]

    # 2. 处理括号占位符（如“（）”）
    if '（）' in question or '（ ）' in question:
        full = question.replace('（）', answer_text).replace('（ ）', answer_text)
    else:
        full = f"{question} 正确答案：{answer_text}"
    return full

In [20]:
def compute_knowledge_relevance(full_question, knowledge_context):
    """
    计算完整试题与知识上下文的语义相关性得分
    策略：将知识上下文按换行拆分为句子列表，计算每句与完整试题的余弦相似度，
          取相似度最高的 top-3 句的平均值（若少于3句则全取平均）。
    返回: 浮点数相关性得分
    """
    if not knowledge_context or pd.isna(knowledge_context):
        return 0.0
    # 按换行分割，过滤空行
    sentences = [s.strip() for s in knowledge_context.split('。') if s.strip()]
    if not sentences:
        return 0.0

    # 编码
    q_emb = embedder.encode(full_question, convert_to_tensor=True)
    s_embs = embedder.encode(sentences, convert_to_tensor=True)

    # 计算余弦相似度
    cos_scores = util.cos_sim(q_emb, s_embs)[0].cpu().numpy()

    # 取 top-3 平均值（或全部平均如果少于3）
    k = min(3, len(sentences))
    top_k_idx = np.argsort(cos_scores)[-k:]
    relevance = cos_scores[top_k_idx].mean()
    return float(relevance)

In [296]:
# 单次任务
class RelevanceEvaluator:
    def evaluate(self, question, context, cutoff=None):
        """
        评估题目与目标知识点的相关性。

        参数:
            question (str): 纯题目文本（无需解析分号分隔字段）
            target_kp (str): 目标知识点
            cutoff (float): 阈值

        返回:
            (float, str): 相关性得分和说明信息
        """
        # 直接使用传入的题目文本和目标知识点计算相关性
        score = compute_knowledge_relevance(question, context)
        return score

        # if score > cutoff:
        #     return score, '相关性良好'
        # else:
        #     return score, f'对于目标知识点{target_kp}的相关性不够'

In [33]:
# ========== 使用示例 ==========
filename="SFT_EVAL/output_sft_2_3_0.txt"
evaluate_relevance_from_txt(filename, filename[:-4]+'_rel'+'.txt')

知识点相关性得分列表：
[0.846, 0.7703, 0.7902, 0.8822, 0.8464, 0.8069, 0.8545, 0.8882, 0.8073, 0.8741, 0.7874, 0.8889, 0.6238, 0.8377, 0.7744, 0.6242, 0.8497, 0.8754, 0.6955, 0.8299, 0.8779, 0.6807, 0.8968, 0.7619, 0.7776, 0.7375, 0.8906, 0.934, 0.6553, 0.7682, 0.8414, 0.8519, 0.8678, 0.756, 0.6782, 0.6486, 0.7928, 0.8705, 0.8574, 0.802, 0.8886, 0.7454, 0.7286, 0.7062, 0.8722, 0.7472, 0.9274, 0.8895, 0.8062, 0.879, 0.7807, 0.7954, 0.8168, 0.8008, 0.9044, 0.8258, 0.8195, 0.9289, 0.6309, 0.8985, 0.8827, 0.8068, 0.8258, 0.7487, 0.8617, 0.8644, 0.8757, 0.9028, 0.6957, 0.8843, 0.8464, 0.8034, 0.8124, 0.8415, 0.8038, 0.8047, 0.8758, 0.8246, 0.78, 0.8265, 0.8142, 0.8315, 0.8427, 0.9202, 0.8303, 0.7009, 0.8846, 0.8419, 0.859, 0.8453, 0.8872, 0.9116, 0.7194, 0.8111, 0.9274, 0.8099, 0.8851, 0.8572, 0.8653, 0.8795]
处理完成，结果已保存至：SFT_EVAL/output_sft_2_3_0_rel.txt


## 2.2 可解释性

In [295]:
def eval_answer(input_txt_path: str, output_txt_path: str, n: int = 5):
    """
    对 .txt 文件中的每道试题，调用 LLM 求解 n 次，记录每次的答案（选项字母），
    追加在原行末尾，以分号分隔，输出新文件。

    输入文件格式（无表头，分号分隔）：
        字段1:知识点, 字段2:难度, 字段3:题型, 字段4:是否运算, 字段5:认知层级,
        字段6:题干, 字段7:选项, 字段8:正确答案, 字段9:知识上下文, 字段10...:其他已有标签
    输出文件格式：原行 + ;答案1;答案2;...;答案n
    """
    with open(input_txt_path, 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f if line.strip()]

    output_lines = []
    for line in lines:
        parts = line.split(';')
        # 提取题干和选项（根据示例：第6字段题干，第7字段选项）
        question_text = parts[5] if len(parts) > 5 else ""
        options_text = parts[6] if len(parts) > 6 else ""

        # 构建完整的选择题文本
        full_question = f"{question_text}\n{options_text}"

        # 为 LLM 构建系统提示和用户提示
        system_prompt = '''
        你是一名资深地理教师，请回答以下选择题。
        只输出选项字母（A、B、C、D）中的一个，不要输出任何其他内容。
        '''
        user_prompt = f"题目：{full_question}\n你的答案："

        
        print(f"\n第{len(output_lines)+1}题",end=' ')
        answers = []
        for i in range(n):
            print(f"第{i+1}次",end=' | ')
            try:
                # 调用已定义好的 llm 变量
                response = llm.invoke([
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ])
                # 获取模型输出文本
                if hasattr(response, 'content'):
                    content = response.content.strip()
                else:
                    content = str(response).strip()
                # 提取第一个 A-D 字母
                match = re.search(r'[A-D]', content)
                answer = match.group(0) if match else 'ERROR'
            except Exception as e:
                print(f"第 {len(output_lines)+1} 行第 {i+1} 次调用出错: {e}")
                answer = 'ERROR'
            
            answers.append(answer)

        # 原行 + 追加的 n 个答案字段
        new_line = line + ';' + ';'.join(answers)
        output_lines.append(new_line)

    # 写入输出文件
    with open(output_txt_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(output_lines))

    print(f"处理完成，共 {len(output_lines)} 道题，每道题记录 {n} 次答案。")
    print(f"输出文件：{output_txt_path}")

In [203]:
# 单次
import re
from collections import Counter

class AnswerablityEvaluator:
    def __init__(self, llm, n=5):
        """
        :param llm: LLM 调用对象（需有 invoke 方法）
        :param n: 作答次数，默认 5
        """
        self.llm = llm
        self.n = n

    def evaluate(self, question, cutoff):
        """
        评估题目答案的一致性（通过多次作答）。
        question: str，纯题目文本（包含题干和选项）
        target_kp: str，目标知识点（仅用于消息提示）
        cutoff: float，阈值（0~1）
        
        return: (float, str) 可解释性得分和建议
        """
        
        system_prompt = '''
        你是一名资深地理教师，请回答以下选择题。
        只输出选项字母（A、B、C、D）中的一个，不要输出任何其他内容。
        '''
        user_prompt = f"题目：{question}\n你的答案："

        answers = []
        for i in range(self.n):
            try:
                response = self.llm.invoke([
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ])
                content = response.content.strip() if hasattr(response, 'content') else str(response).strip()
                match = re.search(r'[A-D]', content)
                answer = match.group(0) if match else 'ERROR'
            except Exception as e:
                print(f"第 {i+1} 次调用出错: {e}")
                answer = 'ERROR'
            answers.append(answer)

        # 计算最高频答案的比例
        counter = Counter(answers)
        most_common_count = counter.most_common(1)[0][1]
        consistency = most_common_count / self.n

        # 根据阈值生成消息
        if consistency > cutoff:
            message = f"答案可解释性良好"
        else:
            message = f"答案的可解释性不够稳定"

        # return consistency, message 
        return consistency

In [23]:
filename2="SFT_EVAL/output_sft_2_1_0_rel.txt"
eval_answer(filename2, filename2[:-4]+'_ans'+'.txt', n=5)


第1题 第1次 | 第 1 行第 1 次调用出错: name 'llm' is not defined
第2次 | 第 1 行第 2 次调用出错: name 'llm' is not defined
第3次 | 第 1 行第 3 次调用出错: name 'llm' is not defined
第4次 | 第 1 行第 4 次调用出错: name 'llm' is not defined
第5次 | 第 1 行第 5 次调用出错: name 'llm' is not defined

第2题 第1次 | 第 2 行第 1 次调用出错: name 'llm' is not defined
第2次 | 第 2 行第 2 次调用出错: name 'llm' is not defined
第3次 | 第 2 行第 3 次调用出错: name 'llm' is not defined
第4次 | 第 2 行第 4 次调用出错: name 'llm' is not defined
第5次 | 第 2 行第 5 次调用出错: name 'llm' is not defined

第3题 第1次 | 第 3 行第 1 次调用出错: name 'llm' is not defined
第2次 | 第 3 行第 2 次调用出错: name 'llm' is not defined
第3次 | 第 3 行第 3 次调用出错: name 'llm' is not defined
第4次 | 第 3 行第 4 次调用出错: name 'llm' is not defined
第5次 | 第 3 行第 5 次调用出错: name 'llm' is not defined

第4题 第1次 | 第 4 行第 1 次调用出错: name 'llm' is not defined
第2次 | 第 4 行第 2 次调用出错: name 'llm' is not defined
第3次 | 第 4 行第 3 次调用出错: name 'llm' is not defined
第4次 | 第 4 行第 4 次调用出错: name 'llm' is not defined
第5次 | 第 4 行第 5 次调用出错: name 'llm' is not defined

第5题 第1次 | 第 5 行第 1 

## 2.3难度准确性

In [25]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

In [26]:
def eval_difficulty_accuracy(generated_txt, bank_xlsx, section, output_txt=None):
    """
    评估生成试题的难度准确性。
    参数:
        generated_txt : str  - 生成试题的 .txt 文件路径（分号分隔，无表头）
        bank_xlsx    : str  - 试题库 .xlsx 文件路径（包含表头，列顺序见说明）
        section      : str  - 生成试题所属的节（如“流水地貌”），用于筛选题库
        output_txt   : str  - 输出文件路径，默认为原文件名加“_dif”
    输出:
        新 .txt 文件，每行末尾追加一列难度准确性得分（相似度），保留4位小数。
    """
    if output_txt is None:
        output_txt = generated_txt.replace('.txt', '_dif.txt')

    # 加载语义相似度模型（CPU 可用）
    

    # ========== 读取题库 ==========
    df = pd.read_excel(bank_xlsx, dtype=str)  # 全部按字符串读取
    bank_items = []  # 每个元素为 (节, 难度, 向量)

    for idx, row in df.iterrows():
        # 按列索引取字段（根据图片列顺序）
        sec = str(row.iloc[1]) if pd.notna(row.iloc[1]) else ''      # 第2列：节
        diff = str(row.iloc[9]) if pd.notna(row.iloc[9]) else ''     # 第10列：难度
        # 组合完整试题文本：材料 + 提问 + 选项1~4
        parts = [
            str(row.iloc[2]) if pd.notna(row.iloc[2]) else '',  # 材料
            str(row.iloc[3]) if pd.notna(row.iloc[3]) else '',  # 提问
            str(row.iloc[4]) if pd.notna(row.iloc[4]) else '',  # 选项1
            str(row.iloc[5]) if pd.notna(row.iloc[5]) else '',  # 选项2
            str(row.iloc[6]) if pd.notna(row.iloc[6]) else '',  # 选项3
            str(row.iloc[7]) if pd.notna(row.iloc[7]) else '',  # 选项4
        ]
        full_text = ' '.join([p for p in parts if p.strip()])
        if not full_text.strip():
            continue  # 跳过空题
        # 计算归一化向量（点积即余弦相似度）
        emb = model.encode(full_text, normalize_embeddings=True)
        bank_items.append((sec, diff, emb))

    # ========== 处理生成试题 ==========
    with open(generated_txt, 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f if line.strip()]

    out_lines = []
    for line in lines:
        parts = line.split(';')
        if len(parts) < 7:
            print(f"警告：行字段不足，跳过 - {line[:50]}")
            out_lines.append(line + ';')  # 补一个空分号
            continue

        diff_gen = parts[1]                     # 生成难度（第2字段）
        q_text = parts[5]                       # 题干（第6字段）
        opts = parts[6]                          # 选项（第7字段）
        full_q = q_text + ' ' + opts

        q_emb = model.encode(full_q, normalize_embeddings=True)

        # 筛选题库中相同节且相同难度的题目
        candidates = [item for item in bank_items if item[0] == section and item[1] == diff_gen]
        if not candidates:
            score = 0.0
        else:
            # 计算与所有候选向量的余弦相似度，取最大值
            bank_embs = np.array([item[2] for item in candidates])
            sims = np.dot(bank_embs, q_emb)      # 所有点积（已归一化）
            score = float(np.max(sims))

        new_line = line + f';{score:.4f}'
        out_lines.append(new_line)

    # ========== 写入输出文件 ==========
    with open(output_txt, 'w', encoding='utf-8') as f:
        f.write('\n'.join(out_lines))

    print(f"难度准确性评估完成，结果已保存至：{output_txt}")

In [202]:
# 单次
class DifficultyEvaluator:
    def __init__(self, bank_xlsx, model):
        """
        初始化评估器，加载题库并预计算向量。

        参数:
            bank_xlsx : str   - 试题库 .xlsx 文件路径
            model     : obj   - 语义相似度模型，需要有 encode 方法
        """
        self.model = model
        self.bank_items = []  # 每个元素为 (节, 难度, 向量)

        # 读取题库
        df = pd.read_excel(bank_xlsx, dtype=str)
        for idx, row in df.iterrows():
            sec = str(row.iloc[1]) if pd.notna(row.iloc[1]) else ''      # 第2列：节
            diff = str(row.iloc[9]) if pd.notna(row.iloc[9]) else ''     # 第10列：难度
            parts = [
                str(row.iloc[2]) if pd.notna(row.iloc[2]) else '',  # 材料
                str(row.iloc[3]) if pd.notna(row.iloc[3]) else '',  # 提问
                str(row.iloc[4]) if pd.notna(row.iloc[4]) else '',  # 选项1
                str(row.iloc[5]) if pd.notna(row.iloc[5]) else '',  # 选项2
                str(row.iloc[6]) if pd.notna(row.iloc[6]) else '',  # 选项3
                str(row.iloc[7]) if pd.notna(row.iloc[7]) else '',  # 选项4
            ]
            full_text = ' '.join([p for p in parts if p.strip()])
            if not full_text.strip():
                continue
            emb = self.model.encode(full_text, normalize_embeddings=True)
            self.bank_items.append((sec, diff, emb))

    def evaluate(self, question_text, difficulty, target_kp, cutoff):
        """
        评估单道试题的难度准确性（纯文本输入版）。

        参数:
            question_text : str   - 完整的题目文本（题干+选项），例如 "下列现象中...\nA.昼夜交替 B.四季变化 ..."
            difficulty    : str   - 该试题的难度标签，如 "中等"
            target_kp     : str   - 目标知识点（节），用于筛选题库中的相同节
            cutoff        : float - 阈值（0~1）

        返回:
            (float, str)  - 难度准确性得分（语义相似度最大值），以及一句建议
        """
        # 难度标签直接使用传入参数
        diff_gen = difficulty
        # 完整试题文本直接使用传入参数
        full_q = question_text

        # 计算当前试题的向量
        q_emb = self.model.encode(full_q, normalize_embeddings=True)

        # 筛选题库中相同节且相同难度的题目
        candidates = [item for item in self.bank_items
                      if item[0] == target_kp and item[1] == diff_gen]
        if not candidates:
            score = 0.0
            message = f"题库中未找到与目标知识点 '{target_kp}' 且难度 '{diff_gen}' 匹配的试题"
        else:
            bank_embs = np.array([item[2] for item in candidates])
            sims = np.dot(bank_embs, q_emb)
            score = float(np.max(sims))
            message = ""

        # 根据阈值生成最终建议
        if score > cutoff:
            final_msg = f"难度准确性良好，当前试题难度符合要求"
        else:
            final_msg = f"难度准确性不足，不符合所设定的要求"

        # return score, final_msg
        return score

In [41]:
filename3="SFT_EVAL/output_instruct_2_3_0_rel_ans.txt"

eval_difficulty_accuracy(
    generated_txt=filename3,
    bank_xlsx='./data/题库 _已标注.xlsx',
    section='喀斯特、海岸和冰川地貌',          # 根据生成试题所属的节填写
    output_txt=filename3[:-4]+'_dif'+'.txt'    # 可选
    )

难度准确性评估完成，结果已保存至：SFT_EVAL/output_instruct_2_3_0_rel_ans_dif.txt


## 3 短期记忆

In [286]:
class SessionMemory: 
    '''用在单次任务里'''
    def __init__(self):
        self.retry_count=0
        self.memory_rel=[]
        self.memory_ans=[]
        self.memory_diff=[]
        self.actions=[]
        self.feedbacks=[]
        self.question=''
        self.kp=''
        self.difficulty=''
        self.context=''

    def add_score(self,evaluator,score):
        '''记录反馈'''
        if evaluator=='rel':
            self.memory_rel.append(score)
        elif evaluator=='ans':
            self.memory_ans.append(score)
        elif evaluator=='diff': 
            self.memory_diff.append(score)
        

    def get_last_score(self,evaluator):
        '''取出上一个score'''
        if self.retry_count == 0:
            return None
        if evaluator=='relevance':
            return self.memory_rel[-1]
        elif evaluator=='answerablity':
            return self.memory_ans[-1]
        elif evaluator=='difficulty': 
            return self.memory_diff[-1]

    def get_last_feedback(self):
        if self.retry_count>0:
            return self.feedbacks[-1]
        return None

    def update_question(self,question): 
        '''更新当前的试题'''
        self.question=question
        self.retry_count+=1

## 4 决策

In [305]:
class Decision:
    def __init__(self):
        self.max_retries=5

    def decide(self,scores,memory):
        '''
        input:评估器打分、feedback
        output: action,具体指令
        actions set: 通过；加强相关性；加强可解释性；加强准确性；丢弃
        '''
        if scores.get('relevance',1.0)<0.7:
            if memory.retry_count <= self.max_retries:
                return 'relevance',f"请阅读以下参考知识，特别加强试题与以下参考知识的相关性。参考知识：{memory.context}"
            else:
                return 'drop','相关性无可救药'

        if scores.get('answerablity',1.0)<0.7:
            if memory.retry_count <= self.max_retries:
                return 'answerablity',f"当前试题的可解释性不够，从题干不能严格地得到唯一正确答案，需要加强"
            else:
                return 'drop','可解释性无可救药'

        if scores.get('difficulty',1.0)<0.5 or scores.get('difficulty',1.0)>0.9:
            if memory.retry_count <= self.max_retries:
                return 'difficulty',f"当前试题的难度不符合要求，需要调整难度"
            else:
                return 'drop','难度准确性无可救药'

        return 'pass','所有评估通过'

## 5 长期记忆

In [32]:
class LongTermMemory:
    """存储到知识图谱节点的元数据里"""
    def __init__(self):
        # 模拟知识图谱节点数据
        self.kp_metadata = {
            "光合作用": {
                "common_omissions": [],           # 常见遗漏知识点
                "inconsistency_risk": 0.2,        # 答案不一致风险
                "difficulty_distribution": [0.3, 0.5, 0.2]  # 简单/中/难比例
            },
            "牛顿第三定律": {
                "common_omissions": ["作用力与反作用力方向"],
                "inconsistency_risk": 0.5,
                "difficulty_distribution": [0.2, 0.4, 0.4]
            }
        }

    def get_metadata(self, kp_name):
        """获取知识点的元数据"""
        return self.kp_metadata.get(kp_name, {})

    def update_metadata(self, kp_name: str, updates):
        """更新知识点的元数据（成功出题后或人工介入后调用）"""
        if kp_name not in self.kp_metadata:
            self.kp_metadata[kp_name] = {}
        self.kp_metadata[kp_name].update(updates)
        print(f"[长期记忆] 更新知识点 {kp_name} 的元数据: {updates}")

## 带决策+记忆的生成agent

In [178]:
answer_propmt=PromptTemplate.from_template(
    '''
    你是一名高中地理资深命题教师。
    
    【已知条件】
    - 知识上下文：
    {context}
    - 目标知识点： {knowledge_point}
    - 难度等级：{difficulty}
    - 题型：{q_type}
    - 是否涉及运算：{calculation}
    - 认知层级（如：记忆 / 理解 / 应用 / 分析 / 评价 / 创造）：
    {cognitive_level}
    
    【出题要求】
    1. 题目必须以「目标知识点」为核心  
    2. 可合理使用上下文中的相关知识点作为支撑，但只能起辅助作用  
    3. 严格匹配给定的题型、难度、是否运算、认知层级
    4. 题干表述清晰，不引入无关信息。纯文本题，不要用到图片。
    5. 设问方式灵活，所问的可能是下一句或者选择一个陈述句（这两种情况最多），也可能是挖空，也可能是从下列选择或排序等
    5. 有概率用真实地理景观为例出题，但不要只用上下文中的，可以用其他的，比如“乞拉朋齐……据此请分析……的成因主要是”
    5. 只生成一道题，不要解释、不输出思路
    6. 要符合高中地理考题风格

    严格按照以下格式（包括换行也要一致）输出，不要含有其他内容如前后缀等，最前面不要有空格
    【输出格式】：
    【{difficulty}题 | {cognitive_level}题 | 运算题（如果涉及运算的话）】前两个xx分别填难度、认知层级
    【题目】
    （另起一行）题目内容
    （如是选择题）【选项】
    【参考答案】
    （另起一行）（如果是选择题，只输出选项一个字母）
    '''
)

In [275]:
revise_prompt=PromptTemplate.from_template(
    '''
    你是一位资深学科教师，上一版题目未通过评估，请根据反馈进行修改。

    【上一版题目】
    {previous_question}
    
    【评估反馈】
    - 问题类型：{action}
    - 具体问题：{feedback}
    
    【修改要求】
    0. 不要给出任何“修改说明”或者标记，直接输出题目就行！！！
    1. 保留上一版题目中正确的部分，不要全盘重写
    2. 仅针对上述“具体问题”进行修正
    3. 修正后仍需覆盖目标知识点：{knowledge_point}
    '''
)

In [306]:
class QuestionGenerator:
    """出题Agent，根据目标单元和上下文生成题目"""
    def __init__(self, long_term_mem: LongTermMemory,memory:SessionMemory):
        self.ltm = long_term_mem
        self.memory=memory

    def generate(self, block,stu_group=0) -> str:
        """生成试题，可结合短期记忆和长期记忆"""
        # 从长期记忆获取该知识点的元数据
        # metadata = self.ltm.get_metadata(target_kp)
        
        if self.memory.retry_count==0: # 初次生成，用base提示词
            prompt = answer_prompt
            try:
                answer, features, context = get_questions(
                    block=block,
                    stu_group=stu_group,
                    knowledge_point=None,
                    prompt=prompt
                )
                features = parse(answer, features, context) 
                
                print('成功')
                # 把第一轮生成过程得到的常量写进短期记忆，包括目标知识点、难度等级、知识上下文 等
                self.memory.kp=features[0]
                self.memory.difficulty=features[1]
                self.memory.context=context
                features = [answer, features, context]
                # print(features)
                return features
                
            except Exception as e:
                print(f'【生成有误，重试】错误信息：{e}')
            
        else: #不是初次生成，说明要修改，要调用短期记忆、修改模版
            prompt=revise_prompt
            # 加入短期记忆中的反馈
            evaluator=self.memory.actions[-1]
            last_feedback = self.memory.get_last_feedback()
            
            answer, features, context = self.get_questions(
                    previous_question=self.memory.question,
                    action=self.memory.actions[-1],
                    feedback=last_feedback,
                    prompt=revise_prompt,
                    knowledge_point=None,
                )
            
            return answer
            
            
        # 生成试题

    def rag_revise(self,knowledge_point,previous_question,action,feedback,prompt):
        # 检索出的子图
        graph_results=retrieve_1hop_subgraph(graph,knowledge_point)
        # 子图->知识上下文
        context=get_knowledge_context(graph_results)
        # 制作完整prompt
        prompt=prompt.format(
                previous_question=previous_question,
                action=action,
                feedback=feedback,
                knowledge_point=knowledge_point
            )
        print(prompt[:20])
        response=llm.invoke(prompt)
    
        return graph_results,context,response.content

    def get_questions(self,knowledge_point,previous_question,action,feedback,prompt):
        # 生成
        graph_results,context,answer=self.rag_revise(
                previous_question=previous_question,
                action=action,
                feedback=feedback,
                knowledge_point=knowledge_point,
                prompt=prompt
            )
            
        features=[knowledge_point,previous_question,action,feedback]
    
        return answer,features,context

In [268]:
ltm=LongTermMemory()
memory=SessionMemory()
gen=QuestionGenerator(ltm,memory)
gen.generate(block='流水地貌')

简单 不涉及 理解


KeyboardInterrupt: 

In [166]:
gen.memory.kp

'山洪'

In [133]:
gen.memory.update_question('在河流自由摆动的平原地区，随着流水对河岸的冲刷与侵蚀，\
                    河流愈来愈弯曲，最后导致河流自然裁弯取直，河水由取直部位径直流去，原来弯曲的河道被废弃，形成湖泊。\
                    因这种湖泊的形状恰似牛轭，故称之为牛轭湖。下列地形区中，牛轭湖最为常见的是？ /n \
                    A. 云贵高原 B. 黄土高原 C. 长江中下游平原 D. 内蒙古高原')
gen.memory.memory.append('答案不稳定，请增强从题干得到答案的唯一明确性')

## 完整pipeline

In [142]:
llm=llm_ds
bank_xlsx='./data/题库 _已标注.xlsx'
block='流水地貌'

In [145]:
# model = SentenceTransformer('all-MiniLM-L6-v2')

In [307]:
def main():
    # 初始化组件
    ltm = LongTermMemory()    #长期记忆
    memory = SessionMemory()  #短期记忆
    generator = QuestionGenerator(ltm,memory)  #生成agent
    relevance_eval = RelevanceEvaluator()    #相关性评估器
    answerablity_eval = AnswerablityEvaluator(llm)   #可解释性评估器
    difficulty_eval = DifficultyEvaluator(bank_xlsx,model)   #难度准确性评估器
    decision = Decision()  #决策agent
    
    final_question = None

    # 开始一次出题任务
    # 最大行动次数为3
    while generator.memory.retry_count <= decision.max_retries:
        print(f"\n--- 第 {generator.memory.retry_count + 1} action ---")
        
        # 1. 生成初始的题目
        if generator.memory.retry_count==0:
            question = generator.generate(block)[1]
            question=question[5]+question[6]
        else:
            question=generator.generate(block)
        generator.memory.update_question(question)
        print(f"生成题目：{question}")

        # 2. 评估 得到分数
        rel_score = relevance_eval.evaluate(question,context=generator.memory.context,cutoff=0.8)
        con_score = answerablity_eval.evaluate(question,cutoff=0.94)
        diff_score = difficulty_eval.evaluate(question,
                                                       difficulty=generator.memory.difficulty,
                                                       target_kp=generator.memory.kp,
                                                       cutoff=0.72)
        
        scores = {"relevance": rel_score, "answerablity": con_score, "difficulty": diff_score}
        # feedbacks = {"relevance": rel_fb, "consistency": con_fb, "difficulty": diff_fb}

        print(f"相关性：{rel_score:.3f}")
        print(f"一致性：{con_score:.2f}")
        print(f"难度：{diff_score:.3f}")
        
        # 3. 更新短期记忆
        generator.memory.add_score("relevance", rel_score)
        generator.memory.add_score("answerablity", con_score)
        generator.memory.add_score("difficulty", diff_score)

        # 4. 决策 得到动作、消息
        action, msg = decision.decide(scores,generator.memory)
        print(f"决策动作：{action}，消息：{msg}")

        # 执行动作
        if action == "pass": # 当前题通过，写入长期记忆（示例：更新易错点、难度分布）
            final_question = question
            # ltm.update_metadata(target_kp, {
            #     "last_success_template": question,
            #     "difficulty_fitted": diff_score
            # })
            break
        elif action in {"relevance",'answerablity','difficulty'}:
            # generator.memory.retry_count += 1
            generator.memory.actions.append(action)
            generator.memory.feedbacks.append(msg)
            
        elif action == "adjust_and_retry":
            # generator.memory.retry_count += 1
            generator.memory.actions.append(action)
            # 这里可以实际修改参数，本例中generator的提示词会读取memory.adjusted_params
            # 实际中可将参数传给生成器
        elif action == "human_intervention":
            print("需要人工介入")
            # 可以在此将问题记录到长期记忆，供人工处理
            break

    if final_question:
        print(f"\n最终通过题目：{final_question}")
    else:
        print("\n未成功出题，需人工处理")


In [311]:
if __name__ == "__main__":
    main()


--- 第 1 action ---
简单 不涉及 分析
成功
生成题目：虎跳峡位于中国西南地区，是世界上最深的峡谷之一，以水流湍急、峡谷幽深著称。峡谷两岸的岩层陡峭，谷底狭窄，江心有一巨石（虎跳石）矗立。该地貌的形成与下列哪一因素关系最为密切？A. 冰川的刨蚀作用 B. 地壳的持续抬升 C. 流水的溯源侵蚀 D. 风力的长期吹蚀
相关性：0.739
一致性：1.00
难度：0.000
决策动作：difficulty，消息：当前试题的难度不符合要求，需要调整难度

--- 第 2 action ---

    你是一位资深学科教师，上一版题
生成题目：虎跳峡位于中国西南地区，是世界上最深的峡谷之一，以水流湍急、峡谷幽深著称。峡谷两岸的岩层陡峭，谷底狭窄，江心有一巨石（虎跳石）矗立。该地貌的形成与下列哪一因素关系最为密切？
A. 冰川的刨蚀作用
B. 地壳的持续抬升与流水的强烈下切侵蚀
C. 流水的溯源侵蚀
D. 风力的长期吹蚀
相关性：0.747
一致性：1.00
难度：0.000
决策动作：difficulty，消息：当前试题的难度不符合要求，需要调整难度

--- 第 3 action ---

    你是一位资深学科教师，上一版题
生成题目：虎跳峡位于中国西南地区，是世界上最深的峡谷之一，以水流湍急、峡谷幽深著称。峡谷两岸的岩层陡峭，谷底狭窄，江心有一巨石（虎跳石）矗立。

**问题：** 关于虎跳峡地貌的形成过程，以下描述最准确的是？

**A.** 该峡谷的形成主要经历了冰川的刨蚀作用，塑造了其U型谷的特征。
**B.** 该峡谷是地壳阶段性抬升，河流间歇性下切，并在特定岩性条件下形成的典型深切峡谷。
**C.** 该峡谷的形成主要依赖于流水的溯源侵蚀作用，导致峡谷不断向上游延伸。
**D.** 该峡谷的形态主要是由风力的长期吹蚀和磨蚀作用塑造的。
相关性：0.555
一致性：1.00
难度：0.000
决策动作：relevance，消息：请阅读以下参考知识，特别加强试题与以下参考知识的相关性。参考知识：流水侵蚀地貌形成于虎跳峡。

--- 第 4 action ---

    你是一位资深学科教师，上一版题
生成题目：虎跳峡位于中国西南地区，是世界上最深的峡谷之一，以水流湍急、峡谷幽深著称。峡谷两岸的岩层陡峭，谷底狭窄，江心有一巨石（虎跳石）矗立

# 